# **3. Construção das matrizes de genótipos**

Este notebook transforma arquivos VCF em representações tabulares adequadas para as etapas posteriores de análise exploratória e Machine Learning.

São tratados dois conjuntos de dados independentes:

1. **Dataset principal - Parsnp**

    O arquivo `parsnp.vcf`, produzido no notebook `02_generate_parsnp_vcf.ipynb`, contém as variantes identificadas a partir do alinhamento de core genome dos genomas NCBI aprovados e dos genomas adicionados
    manualmente pelo laboratório.

   A partir desse VCF serão produzidos:

   - `X_genotypes.parquet`: matriz amostra × locus;
   - `snp_info.parquet`: descrição dos loci;
   - `sample_metadata.parquet`: metadados associados às amostras.

   Essa será a matriz utilizada como entrada principal do pipeline de ML.

2. **Painel externo - mapping**

   Um segundo conjunto foi produzido externamente a partir do mapeamento de reads contra o mesmo genoma de referência utilizado no pipeline principal. Os VCFs
   individuais foram posteriormente combinados em um único arquivo.

   Esse conjunto é mantido separado do dataset Parsnp porque foi obtido por um protocolo diferente de identificação de variantes.

   Ele será convertido em uma segunda matriz de genótipos, destinada inicialmente à **validação externa**, ao **perfilamento alélico** e ao **classificador por assinatura
   alélica**.


A estrutura relevante é:

```text
data/
├── processed/
│   └── v4/
│       └── sample_metadata_clean.parquet
│
├── external/
│   └── mapping_panel/
│       ├── cauris_mapping.vcf
│       └── IDs Cauris.xlsx
│
└── results/
    └── v4/
        ├── parsnp/
        │   └── parsnp.vcf
        │
        └── matrices/
            ├── parsnp/
            │   ├── X_genotypes.parquet
            │   ├── snp_info.parquet
            │   └── sample_metadata.parquet
            │
            └── mapping_panel/
                ├── X_genotypes.parquet
                ├── snp_info.parquet
                └── sample_metadata.parquet

## **3.1 Imports**

In [1]:
import re

import numpy as np
import pandas as pd

from clade_clustering.paths import (
    PROCESSED_DIR,
    RESULTS_DIR,
    EXTERNAL_DIR,
)

## **3.2 Paths config**

In [2]:
DATA_VERSION = "v4"

# Dataset principal
PROCESSED_VERSION_DIR = PROCESSED_DIR / DATA_VERSION
PARSNP_DIR = RESULTS_DIR / DATA_VERSION / "parsnp"

PARSNP_VCF_PATH = PARSNP_DIR / "parsnp.vcf"
SAMPLE_METADATA_CLEAN_PATH = PROCESSED_VERSION_DIR / "sample_metadata_clean.parquet"

# Painel externo
MAPPING_PANEL_DIR = EXTERNAL_DIR / "mapping_panel"
MAPPING_VCF_PATH = MAPPING_PANEL_DIR / "cauris_mapping.vcf"
MAPPING_IDS_PATH = MAPPING_PANEL_DIR / "IDs Cauris.xlsx"

# Saídas
MATRICES_DIR = RESULTS_DIR / DATA_VERSION / "matrices"

PARSNP_MATRICES_DIR = MATRICES_DIR / "parsnp"
MAPPING_MATRICES_DIR = MATRICES_DIR / "mapping_panel"

## **3.3 Artifacts path config**

In [3]:
# Parsnp
PARSNP_X_GENOTYPES_PATH = PARSNP_MATRICES_DIR / "X_genotypes.parquet"
PARSNP_SNP_INFO_PATH = PARSNP_MATRICES_DIR / "snp_info.parquet"
PARSNP_SAMPLE_METADATA_PATH = PARSNP_MATRICES_DIR / "sample_metadata.parquet"


# Mapping panel
MAPPING_X_GENOTYPES_PATH = MAPPING_MATRICES_DIR / "X_genotypes.parquet"
MAPPING_SNP_INFO_PATH = MAPPING_MATRICES_DIR / "snp_info.parquet"
MAPPING_SAMPLE_METADATA_PATH = MAPPING_MATRICES_DIR / "sample_metadata.parquet"


## **3.4 Build output directories and validate the files**

In [4]:
for path in [
    PARSNP_MATRICES_DIR,
    MAPPING_MATRICES_DIR,
    EXTERNAL_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

In [5]:
required_inputs = {
    "VCF Parsnp": PARSNP_VCF_PATH,
    "Metadata NCBI": SAMPLE_METADATA_CLEAN_PATH,
}

# Painel externo: opcional, usado apenas na seção 3.5.2
optional_inputs = {
    "VCF mapping": MAPPING_VCF_PATH,
    "IDs mapping": MAPPING_IDS_PATH,
}

missing_inputs = [
    (name, path)
    for name, path in required_inputs.items()
    if not path.exists()
]

if missing_inputs:
    details = "\n".join(
        f"- {name}: {path}"
        for name, path in missing_inputs
    )

    raise FileNotFoundError(
        f"Entradas necessárias não encontradas:\n{details}"
    )

MAPPING_PANEL_AVAILABLE = all(
    path.exists() for path in optional_inputs.values()
)

print("Entradas validadas:")
for name, path in required_inputs.items():
    print(f"{name:15s}: {path}")

print()

if MAPPING_PANEL_AVAILABLE:
    print("Painel externo encontrado:")
    for name, path in optional_inputs.items():
        print(f"{name:15s}: {path}")
else:
    print(
        "Painel externo ausente: a seção 3.5.2 será ignorada.\n"
        f"Arquivos esperados em: {MAPPING_PANEL_DIR}"
    )

Entradas validadas:
VCF Parsnp     : /home/lucas/workspace/projects/clade-clustering/data/results/v4/parsnp/parsnp.vcf
Metadata NCBI  : /home/lucas/workspace/projects/clade-clustering/data/processed/v4/sample_metadata_clean.parquet
VCF mapping    : /home/lucas/workspace/projects/clade-clustering/data/external/mapping_panel/cauris_mapping.vcf
IDs mapping    : /home/lucas/workspace/projects/clade-clustering/data/external/mapping_panel/IDs Cauris.xlsx


## **3.5 Building the genotype matrix**

### **3.5.1 Parsnp genotype matrix**

O VCF produzido pelo HarvestTools é convertido em duas representações complementares:

- `snp_info`: descrição das variantes identificadas;
- `X_genotypes`: matriz amostra × locus utilizada nas etapas posteriores de análise.

Nesta etapa são mantidas apenas variantes com `FILTER = PASS` e SNPs simples.

In [6]:
header = None
header_idx = None

with open(PARSNP_VCF_PATH) as f:
    for i, line in enumerate(f):
        if line.lstrip('"').startswith("#CHROM"):
            header = line.strip().strip('"').split("\t")
            header[0] = "chrom"
            header_idx = i
            break

if header is None:
    raise RuntimeError("Header #CHROM não encontrado no VCF.")

df_vcf = pd.read_csv(
    PARSNP_VCF_PATH,
    sep="\t",
    skiprows=range(header_idx + 1),
    names=header,
    low_memory=False,
)

print("Dimensão do VCF:", df_vcf.shape)
df_vcf.head()

Dimensão do VCF: (168018, 1264)


,chrom,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,GCA_003013715.2_ASM301371v2_genomic.fna.ref,...,GCA_902703455.1_CA13_genomic.fna,GCA_044589395.1_ASM4458939v1_genomic.fna,GCA_061547965.1_ASM61547965v1_genomic.fna,GCA_046251105.1_ASM4625110v1_genomic.fna,GCA_046253425.1_ASM4625342v1_genomic.fna,GCA_049863345.1_ASM4986334v1_genomic.fna,GCA_053175795.1_18Y-165_genome_20251003_genomic.fna,GCA_053222935.1_ASM5322293v1_genomic.fna,GCA_019039295.1_ASM1903929v1_genomic.fna,GCA_046251655.1_ASM4625165v1_genomic.fna
0,CP043531.1,59534,CTAGACAAAC.GGCGCTGTCG,G,A,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
1,CP043531.1,59538,ACAAACGGCG.CTGTCGCATG,C,A,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
2,CP043531.1,59539,CAAACGGCGC.TGTCGCATGT,T,"C,G",40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
3,CP043531.1,59570,TGAAACCAAA.GTTCTTTTAC,G,A,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
4,CP043531.1,59576,CAAAGTTCTT.TTACTAAAAT,T,C,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0


In [68]:
df_vcf.head(30)

,chrom,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,GCA_003013715.2_ASM301371v2_genomic.fna.ref,...,GCA_902703455.1_CA13_genomic.fna,GCA_044589395.1_ASM4458939v1_genomic.fna,GCA_061547965.1_ASM61547965v1_genomic.fna,GCA_046251105.1_ASM4625110v1_genomic.fna,GCA_046253425.1_ASM4625342v1_genomic.fna,GCA_049863345.1_ASM4986334v1_genomic.fna,GCA_053175795.1_18Y-165_genome_20251003_genomic.fna,GCA_053222935.1_ASM5322293v1_genomic.fna,GCA_019039295.1_ASM1903929v1_genomic.fna,GCA_046251655.1_ASM4625165v1_genomic.fna
0,CP043531.1,59534,CTAGACAAAC.GGCGCTGTCG,G,A,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
1,CP043531.1,59538,ACAAACGGCG.CTGTCGCATG,C,A,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
2,CP043531.1,59539,CAAACGGCGC.TGTCGCATGT,T,"C,G",40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
3,CP043531.1,59570,TGAAACCAAA.GTTCTTTTAC,G,A,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
4,CP043531.1,59576,CAAAGTTCTT.TTACTAAAAT,T,C,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
5,CP043531.1,59582,TCTTTTACTA.AAATTACGCT,A,G,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
6,CP043531.1,59804,TACATTGGCT.TCATTTGTTA,T,A,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
7,CP043531.1,59813,TTCATTTGTT.ACTGAACAGA,A,G,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
8,CP043531.1,59820,GTTACTGAAC.AGAGCTCTTG,A,G,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
9,CP043531.1,59821,TTACTGAACA.GAGCTCTTGC,G,A,40,LCB,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
print("Variantes:", len(df_vcf))
print("Amostras:", len(df_vcf.columns) - 9)
print("FILTER:")
print(df_vcf["FILTER"].value_counts(dropna=False))

Variantes: 168018
Amostras: 1255
FILTER:
FILTER
PASS             97034
N                23840
LCB              23701
N:LCB            12826
N:ALN             5637
N:LCB:ALN         1709
ALN               1160
CID:ALN            665
N:CID:ALN          471
LCB:ALN            453
LCB:CID:ALN        270
N:LCB:CID:ALN      164
LCB:CID             44
CID                 42
N:LCB:CID            1
N:CID                1
Name: count, dtype: int64


In [8]:
# Selecionar apenas variantes PASS
df_pass = df_vcf[
    df_vcf["FILTER"].astype(str).str.lower() == "pass"
].copy()

# Manter apenas SNPs simples
mask_snp = (
    df_pass["REF"].astype(str).str.len().eq(1)
    & df_pass["ALT"].astype(str).apply(
        lambda x: all(len(alt) == 1 for alt in x.split(","))
    )
)

df_pass = df_pass[mask_snp].copy().reset_index(drop=True)

print("SNPs válidos:", len(df_pass))
df_pass.head()

SNPs válidos: 97034


,chrom,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,GCA_003013715.2_ASM301371v2_genomic.fna.ref,...,GCA_902703455.1_CA13_genomic.fna,GCA_044589395.1_ASM4458939v1_genomic.fna,GCA_061547965.1_ASM61547965v1_genomic.fna,GCA_046251105.1_ASM4625110v1_genomic.fna,GCA_046253425.1_ASM4625342v1_genomic.fna,GCA_049863345.1_ASM4986334v1_genomic.fna,GCA_053175795.1_18Y-165_genome_20251003_genomic.fna,GCA_053222935.1_ASM5322293v1_genomic.fna,GCA_019039295.1_ASM1903929v1_genomic.fna,GCA_046251655.1_ASM4625165v1_genomic.fna
0,CP043531.1,60452,acttctcctt.ctccgTGAAG,c,"C,T",40,PASS,NaN,GT,1,...,1,1,1,1,1,1,1,1,1,1
1,CP043531.1,60467,TGAAGAGGAT.ATACCCGGGA,A,G,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
2,CP043531.1,60473,GGATATACCC.GGGATATTCA,G,A,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
3,CP043531.1,60476,TATACCCGGG.ATATTCAGGC,A,G,40,PASS,NaN,GT,0,...,1,1,1,1,1,1,0,1,1,1
4,CP043531.1,60482,CGGGATATTC.AGGCAACTCT,A,G,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0


In [26]:
n_pass = (df_vcf["FILTER"].astype(str).str.lower() == "pass").sum()

assert len(df_pass) == n_pass

print("Todos os registros PASS correspondem a SNPs simples.")

Todos os registros PASS correspondem a SNPs simples.


- **Construção de validação do `snp_info`**

In [9]:
snp_info = df_pass[
    ["chrom", "POS", "ID", "REF", "ALT", "QUAL", "FILTER"]
].copy()

snp_info = snp_info.rename(columns={
    "POS": "pos",
    "ID": "id",
    "REF": "ref",
    "ALT": "alt",
    "QUAL": "qual",
    "FILTER": "filter",
})

snp_info["snp_id"] = (
    snp_info["chrom"].astype(str)
    + ":"
    + snp_info["pos"].astype(str)
    + ":"
    + snp_info["ref"].astype(str)
    + ">"
    + snp_info["alt"].astype(str)
)

print("Dimensão snp_info:", snp_info.shape)
snp_info.head()

Dimensão snp_info: (97034, 8)


,chrom,pos,id,ref,alt,qual,filter,snp_id
0,CP043531.1,60452,acttctcctt.ctccgTGAAG,c,"C,T",40,PASS,"CP043531.1:60452:c>C,T"
1,CP043531.1,60467,TGAAGAGGAT.ATACCCGGGA,A,G,40,PASS,CP043531.1:60467:A>G
2,CP043531.1,60473,GGATATACCC.GGGATATTCA,G,A,40,PASS,CP043531.1:60473:G>A
3,CP043531.1,60476,TATACCCGGG.ATATTCAGGC,A,G,40,PASS,CP043531.1:60476:A>G
4,CP043531.1,60482,CGGGATATTC.AGGCAACTCT,A,G,40,PASS,CP043531.1:60482:A>G


In [11]:
if snp_info["snp_id"].duplicated().any():
    raise RuntimeError("Foram encontrados snp_id duplicados.")

print("snp_id únicos:", snp_info["snp_id"].is_unique)

snp_id únicos: True


- **Construção e validação do `X_genotypes`**

In [10]:
sample_cols = df_pass.columns[9:].tolist()

gt_df = df_pass[sample_cols].copy()

# Extrair apenas GT caso existam campos adicionais no FORMAT
gt_df = gt_df.astype(str).apply(
    lambda col: col.str.split(":").str[0]
)

gt_df = gt_df.replace({
    ".": pd.NA,
    "./.": pd.NA,
    ".|.": pd.NA,
})

gt_df = gt_df.apply(pd.to_numeric, errors="coerce")

# VCF: linhas = SNPs, colunas = isolados
# ML: linhas = isolados, colunas = SNPs
X_genotypes = gt_df.T.copy()

X_genotypes.columns = snp_info["snp_id"].values
X_genotypes.index.name = "vcf_sample"

print("Dimensão X_genotypes:", X_genotypes.shape)
X_genotypes.head()

Dimensão X_genotypes: (1255, 97034)


,"CP043531.1:60452:c>C,T",CP043531.1:60467:A>G,CP043531.1:60473:G>A,CP043531.1:60476:A>G,CP043531.1:60482:A>G,CP043531.1:60487:A>G,CP043531.1:60542:G>A,CP043531.1:60573:T>C,CP043531.1:60580:T>A,CP043531.1:60581:C>G,...,CP043536.1:910890:G>A,CP043536.1:910914:C>T,CP043536.1:910917:T>C,"CP043536.1:910935:G>C,T",CP043536.1:910962:A>C,CP043536.1:911019:A>C,CP043536.1:911022:T>A,CP043536.1:911025:G>A,CP043536.1:911150:T>C,CP043536.1:911236:T>C
vcf_sample,,,,,,,,,,,,,,,,,,,,,
GCA_003013715.2_ASM301371v2_genomic.fna.ref,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_053222555.1_ASM5322255v1_genomic.fna,1,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_056555055.1_ASM5655505v1_genomic.fna,1,0,0,1,0,0,0,0,0,0,...,1,0,1,1,1,0,0,0,0,0
GCA_061480175.1_ASM6148017v1_genomic.fna,1,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_061086085.1_ASM6108608v1_genomic.fna,1,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
def extract_accession(sample_name):
    match = re.search(r"(GC[AF]_\d+\.\d+)", str(sample_name))
    if match:
        return match.group(1)
    return pd.NA


sample_lookup = pd.DataFrame({
    "vcf_sample": X_genotypes.index
})

sample_lookup["accession"] = sample_lookup["vcf_sample"].apply(extract_accession)

print("Amostras no VCF:", len(sample_lookup))
print("Sem accession extraído:", sample_lookup["accession"].isna().sum())

sample_lookup.head()

Amostras no VCF: 1255
Sem accession extraído: 11


,vcf_sample,accession
0,GCA_003013715.2_ASM301371v2_genomic.fna.ref,GCA_003013715.2
1,GCA_053222555.1_ASM5322255v1_genomic.fna,GCA_053222555.1
2,GCA_056555055.1_ASM5655505v1_genomic.fna,GCA_056555055.1
3,GCA_061480175.1_ASM6148017v1_genomic.fna,GCA_061480175.1
4,GCA_061086085.1_ASM6108608v1_genomic.fna,GCA_061086085.1


In [14]:
mask_ref = sample_lookup["vcf_sample"].astype(str).str.endswith(".ref")

n_ref = int(mask_ref.sum())

if n_ref != 1:
    raise RuntimeError(
        f"Esperada exatamente uma entrada .ref, encontrada(s): {n_ref}"
    )

X_genotypes = X_genotypes.loc[~mask_ref.values].copy()

sample_lookup = (
    sample_lookup.loc[~mask_ref]
    .copy()
    .reset_index(drop=True)
)

print("Amostras .ref removidas:", n_ref)
print("X_genotypes:", X_genotypes.shape)
print("sample_lookup:", sample_lookup.shape)
print("Sem accession extraído:", sample_lookup["accession"].isna().sum())

Amostras .ref removidas: 1
X_genotypes: (1254, 97034)
sample_lookup: (1254, 2)
Sem accession extraído: 11


In [15]:
sample_lookup["accession"] = (
    sample_lookup["accession"]
    .fillna(sample_lookup["vcf_sample"])
)

if sample_lookup["accession"].duplicated().any():
    duplicated = sample_lookup.loc[
        sample_lookup["accession"].duplicated(keep=False),
        ["vcf_sample", "accession"],
    ]

    display(duplicated)
    raise RuntimeError("Identificadores de amostra duplicados.")

print("Identificadores únicos:", sample_lookup["accession"].is_unique)

Identificadores únicos: True


- **Merge com os metadados**

In [16]:
df_meta_clean = pd.read_parquet(SAMPLE_METADATA_CLEAN_PATH)

print("Metadata:", df_meta_clean.shape)
df_meta_clean.head()

Metadata: (1244, 24)


,accession,organism,assembly_name,strain,assembly_status,assembly_is_atypical,assembly_warnings,assembly_notes,country,coordinates,...,total_sequence_length,size_mb,contig_n50,n_contigs,assembly_level,fna_path,exclude_size_bad,exclude_partial,exclude_atypical,exclusion_reason
0,GCA_001189475.1,Candidozyma auris,ASM118947v1,6684,current,None,None,[superseded by newer assembly for species],india,12.95 N 77.64 E,...,12498763,12.498763,30547,759,Scaffold,/home/lucas/workspace/projects/clade-clusterin...,False,False,False,
1,GCA_002775015.1,Candidozyma auris,Cand_auris_B11221_V1,B11221,current,None,None,[superseded by newer assembly for species],south africa,not collected,...,12741178,12.741178,1467420,23,Scaffold,/home/lucas/workspace/projects/clade-clusterin...,False,False,False,
2,GCA_003013715.2,Candidozyma auris,ASM301371v2,B11220,current,None,None,None,japan,not collected,...,12249773,12.249773,2336890,7,Complete Genome,/home/lucas/workspace/projects/clade-clusterin...,False,False,False,
3,GCA_003014415.1,Candidozyma auris,Cand_auris_B11243,B11243,current,None,None,None,venezuela,not collected,...,12354602,12.354602,77742,282,Scaffold,/home/lucas/workspace/projects/clade-clusterin...,False,False,False,
4,GCA_004287075.1,Candidozyma auris,RCPF-1821,RCPF-1821,current,None,None,None,russia,55.70 N 37.41 E,...,12323460,12.323460,44953,470,Contig,/home/lucas/workspace/projects/clade-clusterin...,False,False,False,


In [17]:
sample_metadata = sample_lookup.merge(
    df_meta_clean,
    on="accession",
    how="left",
)

In [18]:
ncbi_accessions = set(df_meta_clean["accession"])

mask_without_metadata = ~sample_metadata["accession"].isin(ncbi_accessions)

print(
    "Amostras sem metadata NCBI:",
    int(mask_without_metadata.sum())
)

display(
    sample_metadata.loc[
        mask_without_metadata,
        ["vcf_sample", "accession"],
    ]
)

Amostras sem metadata NCBI: 11


,vcf_sample,accession
91,Ca14.fasta,Ca14.fasta
265,SRR9007776.pilon.fasta,SRR9007776.pilon.fasta
378,SRR18325430.pilon.fasta,SRR18325430.pilon.fasta
667,Ca9.fasta,Ca9.fasta
714,Ca10.fasta,Ca10.fasta
863,Ca13.fasta,Ca13.fasta
899,SRR18325431.pilon.fasta,SRR18325431.pilon.fasta
927,SRR18325432.pilon.fasta,SRR18325432.pilon.fasta
1007,Ca12.fasta,Ca12.fasta
1108,Ca11.fasta,Ca11.fasta


- **Alinhando os índices e verificando a estrutura**

In [19]:
X_genotypes.index = sample_lookup["accession"].values
X_genotypes.index.name = "accession"

sample_metadata = sample_metadata.set_index("accession")
sample_metadata = sample_metadata.loc[X_genotypes.index]

print("X_genotypes:", X_genotypes.shape)
print("snp_info:", snp_info.shape)
print("sample_metadata:", sample_metadata.shape)

X_genotypes: (1254, 97034)
snp_info: (97034, 8)
sample_metadata: (1254, 24)


In [20]:
assert X_genotypes.index.is_unique
assert X_genotypes.columns.is_unique
assert sample_metadata.index.is_unique

assert X_genotypes.shape[0] == sample_metadata.shape[0]
assert X_genotypes.shape[1] == snp_info.shape[0]

assert X_genotypes.index.equals(sample_metadata.index)

print("Matriz e metadados consistentes.")

Matriz e metadados consistentes.


- **Salvando e validando os artefatos: `X_genotypes`, `snp_info`, `sample_metadata`**

In [21]:
# Otimizar o tipo numérico da matriz
if X_genotypes.isna().any().any():
    X_to_save = X_genotypes.astype("float32")
else:
    X_to_save = X_genotypes.astype("int8")

X_to_save.to_parquet(PARSNP_X_GENOTYPES_PATH)
snp_info.to_parquet(PARSNP_SNP_INFO_PATH, index=False)
sample_metadata.to_parquet(PARSNP_SAMPLE_METADATA_PATH)

print(f"X_genotypes salvo em     : {PARSNP_X_GENOTYPES_PATH}")
print(f"snp_info salvo em        : {PARSNP_SNP_INFO_PATH}")
print(f"sample_metadata salvo em : {PARSNP_SAMPLE_METADATA_PATH}")

X_genotypes salvo em     : /home/lucas/workspace/projects/clade-clustering/data/results/v4/matrices/parsnp/X_genotypes.parquet
snp_info salvo em        : /home/lucas/workspace/projects/clade-clustering/data/results/v4/matrices/parsnp/snp_info.parquet
sample_metadata salvo em : /home/lucas/workspace/projects/clade-clustering/data/results/v4/matrices/parsnp/sample_metadata.parquet


In [22]:
X_check = pd.read_parquet(PARSNP_X_GENOTYPES_PATH)
snp_check = pd.read_parquet(PARSNP_SNP_INFO_PATH)
meta_check = pd.read_parquet(PARSNP_SAMPLE_METADATA_PATH)

print("X_genotypes:", X_check.shape)
print("snp_info:", snp_check.shape)
print("sample_metadata:", meta_check.shape)

assert X_check.shape[0] == meta_check.shape[0]
assert X_check.shape[1] == snp_check.shape[0]
assert X_check.index.equals(meta_check.index)

print("Artefatos Parsnp salvos e validados.")

X_genotypes: (1254, 97034)
snp_info: (97034, 8)
sample_metadata: (1254, 24)
Artefatos Parsnp salvos e validados.


### **3.5.2 External mapping matrix**

Além do dataset principal construído com Parsnp, está disponível um painel externo contendo isolados previamente associados aos clados de Candida auris.

Esse conjunto foi produzido independentemente do pipeline principal e é fornecido por dois arquivos:

- um VCF contendo as variantes identificadas nas amostras;
- uma tabela associando os identificadores das amostras aos respectivos clados.

Como a origem e a estrutura desse VCF diferem do arquivo produzido pelo Parsnp, sua representação não será assumida a priori.

Antes da construção da matriz de genótipos, serão inspecionados:

- a correspondência entre as amostras do VCF e da tabela de metadados;
- os tipos de variantes presentes;
- a estrutura dos campos de genótipo;
- a ocorrência de valores ausentes.
- A partir dessa caracterização será definida a representação utilizada para construir a matriz do painel externo.

> **Seção opcional.** O painel externo não é distribuído com o repositório. Sem `cauris_mapping.vcf` e `IDs Cauris.xlsx` em `data/external/mapping_panel/`, a célula abaixo encerra o notebook aqui; as matrizes do Parsnp já foram salvas na seção 3.5.1.

In [ ]:
class SkipSection(Exception):
    """Encerra o notebook nesta seção exibindo só a mensagem, sem traceback."""

    def _render_traceback_(self):
        return str(self).splitlines()


if not MAPPING_PANEL_AVAILABLE:
    raise SkipSection(
        "Painel externo ausente: seção 3.5.2 ignorada.\n"
        f"Para construir a matriz externa, coloque cauris_mapping.vcf e "
        f"IDs Cauris.xlsx em {MAPPING_PANEL_DIR}.\n"
        "As matrizes do Parsnp já foram salvas na seção 3.5.1."
    )

print("Painel externo encontrado: construindo a matriz externa.")

#### **3.5.2.1 Metadados do painel**

In [6]:
mapping_metadata = pd.read_excel(MAPPING_IDS_PATH)

print("Dimensão:", mapping_metadata.shape)
print("Colunas:", mapping_metadata.columns.tolist())

mapping_metadata.head()

Dimensão: (30, 2)
Colunas: ['Sample ID', 'Clade']


,Sample ID,Clade
0,SRR3883438,I
1,SRR10851769,I
2,SRR23696691,I
3,SRR23696697,I
4,SRR26666780,I


In [7]:
mapping_metadata = mapping_metadata.rename(columns={
    "Sample ID": "sample_id",
    "Clade": "clade",
})

if mapping_metadata["sample_id"].isna().any():
    raise RuntimeError("Existem sample_id ausentes.")

if mapping_metadata["sample_id"].duplicated().any():
    raise RuntimeError("Existem sample_id duplicados.")

print(mapping_metadata["clade"].value_counts().sort_index())

clade
I      5
II     5
III    5
IV     5
V      5
VI     5
Name: count, dtype: int64


#### **3.5.2.2 Leitura do header do VCF**

In [8]:
mapping_header = None
mapping_header_idx = None

with open(MAPPING_VCF_PATH) as f:
    for i, line in enumerate(f):
        if line.startswith("#CHROM"):
            mapping_header = line.strip().split("\t")
            mapping_header_idx = i
            break

if mapping_header is None:
    raise RuntimeError("Header #CHROM não encontrado no VCF de mapping.")

mapping_sample_cols = mapping_header[9:]

print("Amostras no VCF:", len(mapping_sample_cols))
print(mapping_sample_cols[:10])

Amostras no VCF: 30
['SRR10851769', 'SRR18325429', 'SRR18325430', 'SRR18325431', 'SRR18325432', 'SRR23696691', 'SRR23696697', 'SRR24877248', 'SRR24877249', 'SRR25455197']


In [9]:
vcf_samples = set(mapping_sample_cols)
metadata_samples = set(mapping_metadata["sample_id"])

missing_in_metadata = vcf_samples - metadata_samples
missing_in_vcf = metadata_samples - vcf_samples

print("VCF sem metadata:", missing_in_metadata)
print("Metadata sem VCF:", missing_in_vcf)

if missing_in_metadata or missing_in_vcf:
    raise RuntimeError("VCF e tabela de clados possuem conjuntos de amostras diferentes.")

print("VCF e metadata possuem exatamente as mesmas amostras.")

VCF sem metadata: set()
Metadata sem VCF: set()
VCF e metadata possuem exatamente as mesmas amostras.


#### **3.5.2.3 Inspeção do VCF**

- **Dataframe lendo o arquivo VCF externo diretamente**

In [54]:
df_mapping_vcf = pd.read_csv(
    MAPPING_VCF_PATH,
    sep="\t",
    skiprows=mapping_header_idx,
    low_memory=False,
)

df_mapping_vcf = df_mapping_vcf.rename(columns={
    "#CHROM": "chrom"
})

print("Dimensão do VCF:", df_mapping_vcf.shape)
df_mapping_vcf.head()

Dimensão do VCF: (479869, 39)


,chrom,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,SRR10851769,...,SRR35767701,SRR35767702,SRR35767703,SRR3883438,SRR40721499,SRR40721504,SRR40721505,SRR40721506,SRR40721508,SRR9007776
0,NC_072812.1,408,.,T,G,225.417,PASS,VDB=0.0074869;SGB=-0.692067;MQSBZ=0;MQ0F=0;MQ=...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.
1,NC_072812.1,413,.,C,A,225.421,PASS,VDB=0.337343;SGB=-0.680642;MQSBZ=0;MQ0F=0;MQ=6...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.
2,NC_072812.1,415,.,CACA,C,228.389,PASS,INDEL;IDV=24;IMF=0.96;VDB=0.0015205;SGB=-0.691...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.
3,NC_072812.1,443,.,AT,A,187.416,PASS,INDEL;IDV=52;IMF=0.981132;VDB=9.98651e-05;SGB=...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,"1/1:144,96,0:1,38","1/1:217,172,0:0,57","1/1:173,110,0:1,41","1/1:148,136,0:1,50","1/1:158,78,0:0,26",./.:.:.
4,NC_072812.1,467,.,T,A,228.305,PASS,VDB=0.4578;SGB=-0.693141;RPBZ=2.35716;MQBZ=2.2...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,"1/1:214,21,0:0,7"


> **os exemplos CACA → C e AT → A são indels (ou variações de outra natureza?), enquanto que os demais acima são SNPs!**

> **São 9 colunas comuns do VCF + 30 colunas de amostras/isolado**

- **Máscara para selecionarmos variações que correspondem apenas às SNPs**

In [11]:
df_mapping_vcf[df_mapping_vcf["ALT"].str.contains("*", regex=False)]


,chrom,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,SRR10851769,...,SRR35767701,SRR35767702,SRR35767703,SRR3883438,SRR40721499,SRR40721504,SRR40721505,SRR40721506,SRR40721508,SRR9007776
2029,NC_072812.1,37258,.,GAAAA,"*,G",527.01,PASS,ExcessHet=0;FS=0;MQ=56.9;QD=20.27;SOR=1.22;DP=...,GT:AD:DP:GQ:PL,./.:.:.:.:.,...,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.
2316,NC_072812.1,42220,.,T,"G,C,*",1987.04,PASS,VDB=0.103423;SGB=-0.693146;MQSBZ=-0.54606;MQ0F...,GT:PL:AD:DP:GQ,"1/2:255,207,102,255,0,255,.,.,.,.:0,35,7,.:.:.",...,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,"1/1:255,157,0,.,.,.,.,.,.,.:0,52,.,.:.:.","1/1:255,221,2,.,.,.,.,.,.,.:0,73,.,.:.:.","1/1:255,250,0,.,.,.,.,.,.,.:0,83,.,.:.:.","1/2:255,255,148,255,0,255,.,.,.,.:0,83,17,.:.:.","1/1:255,139,27,.,.,.,.,.,.,.:0,37,.,.:.:.","1/2:255,255,201,255,0,255,.,.,.,.:0,61,16,.:.:.","1/1:255,57,0,.,.,.,.,.,.,.:0,19,.,.:.:."
6374,NC_072812.1,161216,.,A,"G,*",1329.90,PASS,ExcessHet=0;FS=0;MQ=59.85;QD=30.52;SOR=0.859;V...,GT:AD:DP:GQ:PL,./.:.:.:.:.,...,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,"1/1:0,23,.:.:.:255,69,0,.,.,."
6375,NC_072812.1,161219,.,G,"A,*",1367.06,PASS,ExcessHet=0;FS=0;MQ=59.85;QD=33.76;SOR=0.798;V...,GT:AD:DP:GQ:PL,./.:.:.:.:.,...,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,"1/1:0,22,.:.:.:255,66,0,.,.,."
9713,NC_072812.1,276065,.,A,"T,*",1815.04,PASS,ExcessHet=0;FS=0;MQ=60;QD=32.41;SOR=0.922;VDB=...,GT:AD:DP:GQ:PL,./.:.:.:.:.,...,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,"1/1:0,10,.:.:.:254,30,0,.,.,."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470088,NC_072818.1,711879,.,T,"TGG,*",77.97,PASS,ExcessHet=0;FS=0;MQ=59.09;QD=2.23;SOR=0.869;DP...,GT:AD:DP:GQ:PL,./.:.:.:.:.,...,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.
470089,NC_072818.1,711880,.,TCAC,"T,*",69.01,PASS,ExcessHet=0;FS=0;MQ=59.17;QD=1.97;SOR=0.869;DP...,GT:AD:DP:GQ:PL,./.:.:.:.:.,...,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.
470532,NC_072818.1,719190,.,C,"*,T",1493.04,PASS,ExcessHet=0;FS=0;MQ=60;QD=31.11;SOR=1.067;VDB=...,GT:AD:DP:GQ:PL,./.:.:.:.:.,...,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,"2/2:0,.,22:.:.:255,.,.,66,.,0"
471604,NC_072818.1,740974,.,C,"T,*",1393.04,PASS,ExcessHet=0;FS=0;MQ=60;QD=31.66;SOR=1.508;VDB=...,GT:AD:DP:GQ:PL,./.:.:.:.:.,...,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,./.:.:.:.:.,"1/1:0,20,.:.:.:255,60,0,.,.,."


In [12]:
mask_mapping_snp = (
    df_mapping_vcf["REF"].astype(str).str.len().eq(1)
    & df_mapping_vcf["ALT"].astype(str).apply(
        lambda x: all(len(alt) == 1 for alt in x.split(","))
    )
)

print("Variantes totais:", len(df_mapping_vcf))
print("SNPs:", int(mask_mapping_snp.sum()))
print("Indels/Outras variantes:", int((~mask_mapping_snp).sum()))
print(
    "Multialélicas:",
    int(df_mapping_vcf["ALT"].astype(str).str.contains(",").sum())
)

Variantes totais: 479869
SNPs: 439572
Indels/Outras variantes: 40297
Multialélicas: 11963


> **Há 479969 variantes registrados. Muito maior que ~97.000 obtidos pelo Parsnp!**

- **Avaliando padrões por genótipos**

In [57]:
df_mapping_vcf.head()

,chrom,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,SRR10851769,...,SRR35767701,SRR35767702,SRR35767703,SRR3883438,SRR40721499,SRR40721504,SRR40721505,SRR40721506,SRR40721508,SRR9007776
0,NC_072812.1,408,.,T,G,225.417,PASS,VDB=0.0074869;SGB=-0.692067;MQSBZ=0;MQ0F=0;MQ=...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.
1,NC_072812.1,413,.,C,A,225.421,PASS,VDB=0.337343;SGB=-0.680642;MQSBZ=0;MQ0F=0;MQ=6...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.
2,NC_072812.1,415,.,CACA,C,228.389,PASS,INDEL;IDV=24;IMF=0.96;VDB=0.0015205;SGB=-0.691...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.
3,NC_072812.1,443,.,AT,A,187.416,PASS,INDEL;IDV=52;IMF=0.981132;VDB=9.98651e-05;SGB=...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,"1/1:144,96,0:1,38","1/1:217,172,0:0,57","1/1:173,110,0:1,41","1/1:148,136,0:1,50","1/1:158,78,0:0,26",./.:.:.
4,NC_072812.1,467,.,T,A,228.305,PASS,VDB=0.4578;SGB=-0.693141;RPBZ=2.35716;MQBZ=2.2...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,"1/1:214,21,0:0,7"


In [13]:
df_mapping_vcf[mask_mapping_snp].iloc[5]

chrom                                                NC_072812.1
POS                                                          491
ID                                                             .
REF                                                            A
ALT                                                            C
QUAL                                                     228.393
FILTER                                                      PASS
INFO           VDB=2.35573e-05;SGB=-0.693139;MQSBZ=-0.482806;...
FORMAT                                                  GT:PL:AD
SRR10851769                                              ./.:.:.
SRR18325429                                              ./.:.:.
SRR18325430                                   1/1:255,108,0:0,36
SRR18325431                                   1/1:255,123,0:0,41
SRR18325432                                    1/1:255,90,0:1,34
SRR23696691                                              ./.:.:.
SRR23696697              

In [55]:
snp_info

,chrom,pos,id,ref,alt,qual,filter,snp_id
0,CP043531.1,60452,acttctcctt.ctccgTGAAG,c,"C,T",40,PASS,"CP043531.1:60452:c>C,T"
1,CP043531.1,60467,TGAAGAGGAT.ATACCCGGGA,A,G,40,PASS,CP043531.1:60467:A>G
2,CP043531.1,60473,GGATATACCC.GGGATATTCA,G,A,40,PASS,CP043531.1:60473:G>A
3,CP043531.1,60476,TATACCCGGG.ATATTCAGGC,A,G,40,PASS,CP043531.1:60476:A>G
4,CP043531.1,60482,CGGGATATTC.AGGCAACTCT,A,G,40,PASS,CP043531.1:60482:A>G
...,...,...,...,...,...,...,...,...
97029,CP043536.1,911019,CAGAAACGTT.AACTGGGATT,A,C,40,PASS,CP043536.1:911019:A>C
97030,CP043536.1,911022,AAACGTTAAC.TGGGATTCTT,T,A,40,PASS,CP043536.1:911022:T>A
97031,CP043536.1,911025,CGTTAACTGG.GATTCTTAGC,G,A,40,PASS,CP043536.1:911025:G>A
97032,CP043536.1,911150,GTTAGAAGTC.TTTTTGAGTA,T,C,40,PASS,CP043536.1:911150:T>C


In [14]:
print("FILTER:")
print(df_mapping_vcf["FILTER"].value_counts(dropna=False))

print("\nFORMAT:")
print(df_mapping_vcf["FORMAT"].value_counts(dropna=False))

FILTER:
FILTER
PASS    479869
Name: count, dtype: int64

FORMAT:
FORMAT
GT:AD:DP:GQ:PL    257870
GT:PL:AD          185909
GT:PL:AD:DP:GQ     36090
Name: count, dtype: int64


> **Todos os variantes presentes (479869) já são marcados como `PASS`, então não vamos repetir o tratamento de filtragem usado no Parsnp.**

> Existem três padrões em `FORMAT`, cujo cada sigla significa:

- GT: genotype;
- AD: allele depth;
- DP: depth;
- GQ: genotype quality;
- PL: genotype likelihoods em escala Phred.

- **Medindo proporção genótipos ausentes (`./.`) e a contagem de cada código**

In [15]:
mapping_gt = df_mapping_vcf[mapping_sample_cols].astype(str).apply(
    lambda col: col.str.split(":").str[0]
)

gt_counts = mapping_gt.stack().value_counts(dropna=False)

print(gt_counts.head(15))

missing_mask = mapping_gt.isin([".", "./.", ".|."])

print(
    "\nProporção de genótipos ausentes:",
    f"{missing_mask.to_numpy().mean():.2%}",
)

./.    10837615
1/1     3415388
0/1       75520
2/2       44616
1/2        4071
0/2        3665
3/3        2683
2/3        1104
4/4        1064
2/1         992
0/3         900
3/4         850
1/3         720
5/5         537
4/5         511
Name: count, dtype: int64

Proporção de genótipos ausentes: 75.28%


In [53]:
"0/0" in list(gt_counts.index)

False

In [28]:
mapping_gt = df_mapping_vcf[mapping_sample_cols].astype(str).apply(
    lambda col: col.str.split(":").str[0]
)

mapping_gt

,SRR10851769,SRR18325429,SRR18325430,SRR18325431,SRR18325432,SRR23696691,SRR23696697,SRR24877248,SRR24877249,SRR25455197,...,SRR35767701,SRR35767702,SRR35767703,SRR3883438,SRR40721499,SRR40721504,SRR40721505,SRR40721506,SRR40721508,SRR9007776
0,./.,./.,1/1,1/1,1/1,./.,./.,./.,1/1,1/1,...,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.
1,./.,./.,./.,./.,./.,./.,./.,./.,1/1,1/1,...,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.
2,./.,./.,1/1,1/1,1/1,./.,./.,./.,./.,./.,...,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.
3,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.,...,./.,./.,./.,./.,1/1,1/1,1/1,1/1,1/1,./.
4,./.,./.,1/1,1/1,1/1,./.,./.,./.,./.,./.,...,./.,./.,./.,./.,./.,./.,./.,./.,./.,1/1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479864,./.,./.,./.,./.,./.,./.,./.,./.,./.,1/1,...,./.,./.,./.,./.,2/2,2/2,2/2,2/2,2/2,./.
479865,1/1,./.,./.,./.,./.,1/1,1/1,./.,./.,1/1,...,./.,./.,./.,1/1,1/1,1/1,1/1,1/1,1/1,./.
479866,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.,...,./.,1/1,./.,./.,./.,./.,./.,./.,./.,./.
479867,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.,...,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.


In [20]:
mapping_gt.stack()

0       SRR10851769    ./.
        SRR18325429    ./.
        SRR18325430    1/1
        SRR18325431    1/1
        SRR18325432    1/1
                      ... 
479868  SRR40721504    ./.
        SRR40721505    ./.
        SRR40721506    ./.
        SRR40721508    ./.
        SRR9007776     ./.
Length: 14396070, dtype: object

- **Mapeando linhas comuns entre o vcf externo e o parsnp**

In [21]:
df_mapping_vcf.head()

,chrom,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,SRR10851769,...,SRR35767701,SRR35767702,SRR35767703,SRR3883438,SRR40721499,SRR40721504,SRR40721505,SRR40721506,SRR40721508,SRR9007776
0,NC_072812.1,408,.,T,G,225.417,PASS,VDB=0.0074869;SGB=-0.692067;MQSBZ=0;MQ0F=0;MQ=...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.
1,NC_072812.1,413,.,C,A,225.421,PASS,VDB=0.337343;SGB=-0.680642;MQSBZ=0;MQ0F=0;MQ=6...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.
2,NC_072812.1,415,.,CACA,C,228.389,PASS,INDEL;IDV=24;IMF=0.96;VDB=0.0015205;SGB=-0.691...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.
3,NC_072812.1,443,.,AT,A,187.416,PASS,INDEL;IDV=52;IMF=0.981132;VDB=9.98651e-05;SGB=...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,"1/1:144,96,0:1,38","1/1:217,172,0:0,57","1/1:173,110,0:1,41","1/1:148,136,0:1,50","1/1:158,78,0:0,26",./.:.:.
4,NC_072812.1,467,.,T,A,228.305,PASS,VDB=0.4578;SGB=-0.693141;RPBZ=2.35716;MQBZ=2.2...,GT:PL:AD,./.:.:.,...,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,./.:.:.,"1/1:214,21,0:0,7"


In [ ]:
CHROM_MAP = {
    "CP043531.1": "NC_072812.1",
    "CP043532.1": "NC_072813.1",
    "CP043533.1": "NC_072814.1",
    "CP043534.1": "NC_072815.1",
    "CP043535.1": "NC_072816.1",
    "CP043536.1": "NC_072817.1",
    "CP043537.1": "NC_072818.1",
}

parsnp_sites = snp_info[
    ["chrom", "pos", "ref", "alt", "snp_id"]
].copy()

parsnp_sites["chrom_external"] = (
    parsnp_sites["chrom"].map(CHROM_MAP)
)

external_sites = df_mapping_vcf.loc[
    mask_mapping_snp,
    ["chrom", "POS", "REF", "ALT"]
].copy()

# Preserva a linha de origem para recuperar os GT posteriormente
external_sites["vcf_row"] = external_sites.index

matched_sites = parsnp_sites.merge(
    external_sites,
    left_on=["chrom_external", "pos"],  # Junta as linhas se for no mesmo cromossomo e na mesma coordenada
    right_on=["chrom", "POS"],
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("_parsnp", "_vcf"),
)

matched_sites["ref_matches"] = (
    matched_sites["ref"].str.upper()
    == matched_sites["REF"].str.upper()
)

print("Loci Parsnp:", len(parsnp_sites))
print("Posições encontradas:", matched_sites["vcf_row"].notna().sum())
print("REF compatível:", matched_sites["ref_matches"].sum())

display(
    matched_sites[
        ["snp_id", "vcf_row", "ref_matches"]
    ].head()
)

Loci Parsnp: 97034
Posições encontradas: 74336
REF compatível: 74336


,snp_id,vcf_row,ref_matches
0,"CP043531.1:60452:c>C,T",2934.0,True
1,CP043531.1:60467:A>G,2935.0,True
2,CP043531.1:60473:G>A,2936.0,True
3,CP043531.1:60476:A>G,2937.0,True
4,CP043531.1:60482:A>G,2938.0,True


In [32]:
parsnp_sites

,chrom,pos,ref,alt,snp_id,chrom_external
0,CP043531.1,60452,c,"C,T","CP043531.1:60452:c>C,T",NC_072812.1
1,CP043531.1,60467,A,G,CP043531.1:60467:A>G,NC_072812.1
2,CP043531.1,60473,G,A,CP043531.1:60473:G>A,NC_072812.1
3,CP043531.1,60476,A,G,CP043531.1:60476:A>G,NC_072812.1
4,CP043531.1,60482,A,G,CP043531.1:60482:A>G,NC_072812.1
...,...,...,...,...,...,...
97029,CP043536.1,911019,A,C,CP043536.1:911019:A>C,NC_072817.1
97030,CP043536.1,911022,T,A,CP043536.1:911022:T>A,NC_072817.1
97031,CP043536.1,911025,G,A,CP043536.1:911025:G>A,NC_072817.1
97032,CP043536.1,911150,T,C,CP043536.1:911150:T>C,NC_072817.1


In [33]:
external_sites

,chrom,POS,REF,ALT,vcf_row
0,NC_072812.1,408,T,G,0
1,NC_072812.1,413,C,A,1
4,NC_072812.1,467,T,A,4
5,NC_072812.1,488,T,C,5
6,NC_072812.1,489,G,T,6
...,...,...,...,...,...
479861,NC_072818.1,879840,C,G,479861
479862,NC_072818.1,879857,A,T,479862
479863,NC_072818.1,879867,T,C,479863
479864,NC_072818.1,879868,G,"A,C",479864


In [37]:
matched_sites

,chrom_parsnp,pos,ref,alt,snp_id,chrom_external,chrom_vcf,POS,REF,ALT,vcf_row,_merge,ref_matches
0,CP043531.1,60452,c,"C,T","CP043531.1:60452:c>C,T",NC_072812.1,NC_072812.1,60452.0,C,T,2934.0,both,True
1,CP043531.1,60467,A,G,CP043531.1:60467:A>G,NC_072812.1,NC_072812.1,60467.0,A,G,2935.0,both,True
2,CP043531.1,60473,G,A,CP043531.1:60473:G>A,NC_072812.1,NC_072812.1,60473.0,G,A,2936.0,both,True
3,CP043531.1,60476,A,G,CP043531.1:60476:A>G,NC_072812.1,NC_072812.1,60476.0,A,G,2937.0,both,True
4,CP043531.1,60482,A,G,CP043531.1:60482:A>G,NC_072812.1,NC_072812.1,60482.0,A,G,2938.0,both,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
97029,CP043536.1,911019,A,C,CP043536.1:911019:A>C,NC_072817.1,NC_072817.1,911019.0,A,C,429816.0,both,True
97030,CP043536.1,911022,T,A,CP043536.1:911022:T>A,NC_072817.1,NC_072817.1,911022.0,T,A,429817.0,both,True
97031,CP043536.1,911025,G,A,CP043536.1:911025:G>A,NC_072817.1,NC_072817.1,911025.0,G,A,429818.0,both,True
97032,CP043536.1,911150,T,C,CP043536.1:911150:T>C,NC_072817.1,NC_072817.1,911150.0,T,C,429819.0,both,True


In [36]:
matched_sites.ref_matches.sum()

np.int64(74336)

- **Imputação**

In [40]:
X_mapping_arr = np.zeros(
    (len(mapping_sample_cols), len(snp_info)),
    dtype=np.int8,
)

X_mapping_arr

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(30, 97034), dtype=int8)

In [41]:
shared = matched_sites.loc[matched_sites["ref_matches"]]
shared

,chrom_parsnp,pos,ref,alt,snp_id,chrom_external,chrom_vcf,POS,REF,ALT,vcf_row,_merge,ref_matches
0,CP043531.1,60452,c,"C,T","CP043531.1:60452:c>C,T",NC_072812.1,NC_072812.1,60452.0,C,T,2934.0,both,True
1,CP043531.1,60467,A,G,CP043531.1:60467:A>G,NC_072812.1,NC_072812.1,60467.0,A,G,2935.0,both,True
2,CP043531.1,60473,G,A,CP043531.1:60473:G>A,NC_072812.1,NC_072812.1,60473.0,G,A,2936.0,both,True
3,CP043531.1,60476,A,G,CP043531.1:60476:A>G,NC_072812.1,NC_072812.1,60476.0,A,G,2937.0,both,True
4,CP043531.1,60482,A,G,CP043531.1:60482:A>G,NC_072812.1,NC_072812.1,60482.0,A,G,2938.0,both,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
97029,CP043536.1,911019,A,C,CP043536.1:911019:A>C,NC_072817.1,NC_072817.1,911019.0,A,C,429816.0,both,True
97030,CP043536.1,911022,T,A,CP043536.1:911022:T>A,NC_072817.1,NC_072817.1,911022.0,T,A,429817.0,both,True
97031,CP043536.1,911025,G,A,CP043536.1:911025:G>A,NC_072817.1,NC_072817.1,911025.0,G,A,429818.0,both,True
97032,CP043536.1,911150,T,C,CP043536.1:911150:T>C,NC_072817.1,NC_072817.1,911150.0,T,C,429819.0,both,True


In [42]:
mapping_gt

,SRR10851769,SRR18325429,SRR18325430,SRR18325431,SRR18325432,SRR23696691,SRR23696697,SRR24877248,SRR24877249,SRR25455197,...,SRR35767701,SRR35767702,SRR35767703,SRR3883438,SRR40721499,SRR40721504,SRR40721505,SRR40721506,SRR40721508,SRR9007776
0,./.,./.,1/1,1/1,1/1,./.,./.,./.,1/1,1/1,...,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.
1,./.,./.,./.,./.,./.,./.,./.,./.,1/1,1/1,...,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.
2,./.,./.,1/1,1/1,1/1,./.,./.,./.,./.,./.,...,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.
3,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.,...,./.,./.,./.,./.,1/1,1/1,1/1,1/1,1/1,./.
4,./.,./.,1/1,1/1,1/1,./.,./.,./.,./.,./.,...,./.,./.,./.,./.,./.,./.,./.,./.,./.,1/1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479864,./.,./.,./.,./.,./.,./.,./.,./.,./.,1/1,...,./.,./.,./.,./.,2/2,2/2,2/2,2/2,2/2,./.
479865,1/1,./.,./.,./.,./.,1/1,1/1,./.,./.,1/1,...,./.,./.,./.,1/1,1/1,1/1,1/1,1/1,1/1,./.
479866,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.,...,./.,1/1,./.,./.,./.,./.,./.,./.,./.,./.
479867,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.,...,./.,./.,./.,./.,./.,./.,./.,./.,./.,./.


In [43]:
shared = matched_sites.loc[matched_sites["ref_matches"]]
vcf_rows = shared["vcf_row"].astype(int).to_numpy()

gt_values = mapping_gt.loc[
    vcf_rows, mapping_sample_cols
].to_numpy()

gt_values

array([['./.', '1/1', '1/1', ..., './.', './.', '1/1'],
       ['./.', '1/1', '1/1', ..., './.', './.', '1/1'],
       ['./.', '1/1', '1/1', ..., './.', './.', '1/1'],
       ...,
       ['./.', './.', './.', ..., './.', './.', './.'],
       ['./.', '1/1', '1/1', ..., './.', './.', '1/1'],
       ['./.', './.', './.', ..., './.', './.', './.']],
      shape=(74336, 30), dtype=object)

In [48]:
# Baseline: inicializar todos os genótipos como REF (0)
X_mapping_arr = np.zeros(
    (len(mapping_sample_cols), len(snp_info)),
    dtype=np.int8,
)

qc = {
    "missing_gt": np.zeros(len(mapping_sample_cols), dtype=int),
    "mixed_gt": np.zeros(len(mapping_sample_cols), dtype=int),
    "unmapped_allele": np.zeros(len(mapping_sample_cols), dtype=int),
    "spanning_deletion": np.zeros(len(mapping_sample_cols), dtype=int),
    "mapped_gt": np.zeros(len(mapping_sample_cols), dtype=int),
}

shared = matched_sites.loc[matched_sites["ref_matches"]]
vcf_rows = shared["vcf_row"].astype(int).to_numpy()

gt_values = mapping_gt.loc[
    vcf_rows, mapping_sample_cols
].to_numpy()

for row_idx, (_, site) in enumerate(shared.iterrows()):
    col_idx = matched_sites.index.get_loc(site.name)

    parsnp_alleles = [
        site["ref"].upper(),
        *site["alt"].upper().split(","),
    ]

    external_alleles = [
        site["REF"].upper(),
        *site["ALT"].upper().split(","),
    ]

    for sample_idx, gt in enumerate(gt_values[row_idx]):
        parts = re.split(r"[/|]", str(gt))

        if not all(p.isdigit() for p in parts):
            qc["missing_gt"][sample_idx] += 1
            continue

        if len(set(parts)) != 1:
            qc["mixed_gt"][sample_idx] += 1
            continue

        allele_idx = int(parts[0])

        if allele_idx >= len(external_alleles):
            qc["unmapped_allele"][sample_idx] += 1
            continue

        if allele_idx == 0:
            qc["mapped_gt"][sample_idx] += 1
            continue

        base = external_alleles[allele_idx]

        # "*": a amostra tem uma deleção vinda de variante vizinha e não possui
        # base nesta posição. Tratado como ausente (permanece no baseline REF)
        if base == "*":
            qc["spanning_deletion"][sample_idx] += 1
            continue

        # Exige correspondência nucleotídica inequívoca
        matches = [
            j for j, allele in enumerate(parsnp_alleles)
            if allele == base
        ]

        if len(matches) != 1:
            qc["unmapped_allele"][sample_idx] += 1
            continue

        X_mapping_arr[sample_idx, col_idx] = matches[0]
        qc["mapped_gt"][sample_idx] += 1

X_mapping_genotypes = pd.DataFrame(
    X_mapping_arr,
    index=pd.Index(mapping_sample_cols, name="sample_id"),
    columns=snp_info["snp_id"].to_numpy(),
)

print("Matriz externa:", X_mapping_genotypes.shape)
display(pd.DataFrame(qc, index=mapping_sample_cols))


Matriz externa: (30, 97034)


,missing_gt,mixed_gt,unmapped_allele,spanning_deletion,mapped_gt
SRR10851769,64028,0,0,0,10308
SRR18325429,24267,122,33,0,49914
SRR18325430,23631,37,26,0,50642
SRR18325431,23617,23,30,0,50666
SRR18325432,23612,18,25,0,50681
SRR23696691,64051,0,0,0,10285
SRR23696697,64051,1,0,0,10284
SRR24877248,68213,0,0,0,6123
SRR24877249,52189,10,0,0,22137
SRR25455197,45310,1,3,0,29022


| Ordem | Categoria | Quando | Exemplo | Valor em `X_mapping_arr` |
|---|---|---|---|---|
| 1 | `missing_gt` | O GT não tem índices numéricos: sem chamada | `./.` | fica `0` |
| 2 | `mixed_gt` | Os dois índices são diferentes | `0/1`, `1/2` | fica `0` |
| 3 | `unmapped_allele` (a) | O índice aponta além da lista de alelos da linha (GT malformado) | `3/3` numa linha com só 2 ALT | fica `0` |
| 4 | `mapped_gt` (REF) | Índice `0`: a amostra tem a base da referência | `0/0` | fica `0` |
| 5 | `spanning_deletion` | O alelo apontado é `*` | `2/2` em `ALT=G,*` | fica `0` |
| 6 | `unmapped_allele` (b) | A base da amostra não corresponde a exatamente 1 alelo do Parsnp naquele locus | Parsnp tem `A>G`, a amostra tem `T` | fica `0` |
| 7 | `mapped_gt` (ALT) | A base corresponde a um único alelo do Parsnp | Parsnp `A>G`, a amostra tem `G` | recebe o código do Parsnp (`1`) |

> Os testes são aplicados em ordem, por locus e por amostra; o primeiro que se aplica decide a categoria. As categorias somam 74.336 por amostra (loci pareados com REF compatível). Os 22.698 loci do Parsnp ausentes do VCF externo não são contados e permanecem REF (`0`).


In [50]:
X_mapping_genotypes

,"CP043531.1:60452:c>C,T",CP043531.1:60467:A>G,CP043531.1:60473:G>A,CP043531.1:60476:A>G,CP043531.1:60482:A>G,CP043531.1:60487:A>G,CP043531.1:60542:G>A,CP043531.1:60573:T>C,CP043531.1:60580:T>A,CP043531.1:60581:C>G,...,CP043536.1:910890:G>A,CP043536.1:910914:C>T,CP043536.1:910917:T>C,"CP043536.1:910935:G>C,T",CP043536.1:910962:A>C,CP043536.1:911019:A>C,CP043536.1:911022:T>A,CP043536.1:911025:G>A,CP043536.1:911150:T>C,CP043536.1:911236:T>C
sample_id,,,,,,,,,,,,,,,,,,,,,
SRR10851769,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SRR18325429,2,1,1,1,0,0,1,0,1,1,...,0,1,1,0,1,1,0,0,1,0
SRR18325430,2,1,1,1,0,0,1,0,1,1,...,0,1,1,0,1,1,0,0,1,0
SRR18325431,2,1,1,1,0,0,1,0,1,1,...,0,1,1,0,1,1,0,0,1,0
SRR18325432,2,1,1,1,0,0,1,0,1,1,...,0,1,1,0,1,1,0,0,1,0
SRR23696691,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SRR23696697,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SRR24877248,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SRR24877249,0,0,0,1,0,0,1,0,0,0,...,0,0,0,0,0,1,1,1,0,1


In [49]:
mapping_qc_df = pd.DataFrame(
    qc,
    index=X_mapping_genotypes.index
)

mapping_qc_df["called_fraction"] = (
    mapping_qc_df["mapped_gt"] / len(snp_info)
)

mapping_qc_df["n_alt"] = (
    X_mapping_genotypes.ne(0).sum(axis=1)
)

display(
    mapping_qc_df[
        ["mapped_gt", "called_fraction", "n_alt"]
    ].sort_values("called_fraction")
)

,mapped_gt,called_fraction,n_alt
sample_id,,,
SRR35767703,0,0.000000,0
SRR35767701,2,0.000021,2
SRR35767699,22,0.000227,22
SRR35767700,25,0.000258,25
SRR35767702,26,0.000268,26
SRR24877248,6123,0.063102,6123
SRR40721504,10012,0.103180,10012
SRR40721505,10015,0.103211,10015
SRR40721506,10015,0.103211,10015


**Origem de cada célula de `X_mapping_genotypes` (30 × 97.034)**

| Situação do locus / genótipo | Loci envolvidos | Contado no QC como | Valor na matriz |
|---|---|---|---|
| Locus do Parsnp ausente do VCF externo | 22.698 | — (não contado) | `0` (REF imputado) |
| `./.` (sem chamada) | dentro dos 74.336 pareados | `missing_gt` | `0` (REF imputado) |
| Genótipo misto (`0/1`, `1/2`) | dentro dos 74.336 pareados | `mixed_gt` | `0` |
| Alelo `*` (deleção vizinha) | dentro dos 74.336 pareados | `spanning_deletion` | `0` |
| Alelo sem correspondência no Parsnp | dentro dos 74.336 pareados | `unmapped_allele` | `0` |
| `0/0` (REF chamado) | dentro dos 74.336 pareados (inexistente neste VCF) | `mapped_gt` | `0` |
| ALT traduzido para o código do Parsnp | dentro dos 74.336 pareados | `mapped_gt` | código do Parsnp (`1`, `2`, …) |

> A imputação REF não é um passo explícito: a matriz nasce com `np.zeros` e só é sobrescrita na última linha da tabela.

**Colunas de `mapping_qc_df`**

| Coluna | Significado |
|---|---|
| `missing_gt`, `mixed_gt`, `unmapped_allele`, `spanning_deletion`, `mapped_gt` | Contagens por amostra; somam 74.336 |
| `called_fraction` | `mapped_gt / 97.034`; máximo possível ≈ 0,77. Neste VCF mede o quanto a amostra difere da referência (não há `0/0`) |
| `n_alt` | Células ≠ 0 na linha da amostra; aqui igual a `mapped_gt` |


In [36]:
from clade_clustering.paths import RESULTS_DIR

MAPPING_MATRICES_DIR = RESULTS_DIR / "v4" / "matrices" / "mapping_panel"
MAPPING_MATRICES_DIR.mkdir(parents=True, exist_ok=True)

MAPPING_X_GENOTYPES_PATH = MAPPING_MATRICES_DIR / "X_mapping_genotypes.parquet"
MAPPING_QC_PATH = MAPPING_MATRICES_DIR / "genotype_call_qc.csv"

X_mapping_genotypes.to_parquet(MAPPING_X_GENOTYPES_PATH)
mapping_qc_df.to_csv(MAPPING_QC_PATH, index_label="sample_id")

print(f"Matriz externa: {X_mapping_genotypes.shape}")
print(f"Salvo em: {MAPPING_X_GENOTYPES_PATH}")

Matriz externa: (30, 97034)
Salvo em: /home/lucas/workspace/projects/clade-clustering/data/results/v4/matrices/mapping_panel/X_mapping_genotypes.parquet
